# Task 7: Natural Language Processing Basics
### CoreTech Feedback Dataset — NLP Pipeline
**Steps Covered:**
1. Load CSV Dataset
2. Text Cleaning (lowercase, remove punctuation, remove stopwords)
3. Tokenization + Lemmatization
4. TF-IDF Vectorization
5. Sentiment Distribution Bar Chart
6. Word Cloud of Most Common Words
7. Naive Bayes Sentiment Classification + Accuracy

## Step 0: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install wordcloud nltk scikit-learn pandas matplotlib seaborn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from wordcloud import WordCloud

print("✅ All libraries imported successfully!")

## Step 1: Load the Dataset

In [ ]:
# -------------------------------------------------------
# OPTION A: Upload coretech_feedback.csv manually
# Run this cell and upload the file when prompted
# -------------------------------------------------------
from google.colab import files
uploaded = files.upload()  # Upload coretech_feedback.csv here

df = pd.read_csv('coretech_feedback.csv')

print(f"✅ Dataset loaded! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head(10)

In [ ]:
# Basic info about dataset
print("=" * 50)
print("DATASET INFORMATION")
print("=" * 50)
print(df.info())
print("\nSentiment Value Counts:")
print(df['Sentiment'].value_counts())
print("\nRating Statistics:")
print(df['Rating'].describe())

## Step 2: Text Cleaning

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """
    Clean text by:
    1. Converting to lowercase
    2. Removing punctuation
    3. Removing stopwords
    """
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Step 3: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 4: Remove stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]
    
    return ' '.join(words)

# Apply cleaning
df['Cleaned_Text'] = df['Feedback_Text'].apply(clean_text)

print("✅ Text Cleaning Done!")
print("\n--- BEFORE vs AFTER Cleaning (First 3 rows) ---")
for i in range(3):
    print(f"\n[Row {i+1}]")
    print(f"ORIGINAL : {df['Feedback_Text'].iloc[i]}")
    print(f"CLEANED  : {df['Cleaned_Text'].iloc[i]}")

## Step 3: Tokenization + Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()

def tokenize_and_lemmatize(text):
    """
    Tokenize text and apply lemmatization to each token.
    Returns a list of lemmatized tokens.
    """
    # Tokenize
    tokens = word_tokenize(text)
    
    # Lemmatize each token
    lemmatized = [lemmatizer.lemmatize(token) for token in tokens if token.isalpha()]
    
    return lemmatized

# Apply tokenization + lemmatization
df['Tokens'] = df['Cleaned_Text'].apply(tokenize_and_lemmatize)

# Rejoin tokens into a string for vectorization
df['Processed_Text'] = df['Tokens'].apply(lambda x: ' '.join(x))

print("✅ Tokenization + Lemmatization Done!")
print("\n--- Sample Tokens (First 5 rows) ---")
for i in range(5):
    print(f"Row {i+1}: {df['Tokens'].iloc[i]}")

## Step 4: TF-IDF Vectorization

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=100)  # Top 100 features
X = tfidf.fit_transform(df['Processed_Text'])

print("✅ TF-IDF Vectorization Done!")
print(f"\nMatrix Shape: {X.shape}")
print(f"(Rows = {X.shape[0]} feedback messages, Columns = {X.shape[1]} features)")

# Show top 20 TF-IDF feature names
feature_names = tfidf.get_feature_names_out()
print(f"\nTop 20 TF-IDF Features:")
print(list(feature_names[:20]))

# Show TF-IDF matrix as DataFrame (first 5 rows, first 10 features)
tfidf_df = pd.DataFrame(X.toarray(), columns=feature_names)
print("\nTF-IDF Matrix Preview (5 rows x 10 features):")
tfidf_df.iloc[:5, :10]

## Step 5: Sentiment Distribution Bar Chart

In [ ]:
# Count sentiments
sentiment_counts = df['Sentiment'].value_counts()

# Colors for each sentiment
colors = {'Positive': '#2ecc71', 'Neutral': '#f39c12', 'Negative': '#e74c3c'}
bar_colors = [colors.get(s, '#3498db') for s in sentiment_counts.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(sentiment_counts.index, sentiment_counts.values,
               color=bar_colors, edgecolor='black', linewidth=0.8)

# Add count labels on bars
for bar, val in zip(bars, sentiment_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha='center', va='bottom', fontsize=13, fontweight='bold')

plt.title('Sentiment Distribution in CoreTech Feedback', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Sentiment', fontsize=12)
plt.ylabel('Number of Feedbacks', fontsize=12)
plt.ylim(0, sentiment_counts.max() + 3)
plt.xticks(fontsize=12)
plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Sentiment Distribution Chart saved as 'sentiment_distribution.png'")

## Step 6: Word Cloud of Most Common Words

In [ ]:
# Combine all processed text
all_words = ' '.join(df['Processed_Text'].tolist())

# Generate Word Cloud
wordcloud = WordCloud(
    width=900,
    height=500,
    background_color='white',
    colormap='viridis',
    max_words=80,
    min_font_size=10,
    collocations=False
).generate(all_words)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in CoreTech Feedback', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Word Cloud saved as 'wordcloud.png'")

## Step 7: Naive Bayes Sentiment Classification

In [ ]:
# Prepare features (X) and labels (y)
X_vec = tfidf.transform(df['Processed_Text'])  # Use already-fitted TF-IDF
y = df['Sentiment']

# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")

# Train Naive Bayes Model
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Predictions
y_pred = nb_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("\n" + "=" * 50)
print("NAIVE BAYES CLASSIFICATION RESULTS")
print("=" * 50)
print(f"\n✅ Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred, labels=['Positive', 'Neutral', 'Negative'])

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Positive', 'Neutral', 'Negative'],
            yticklabels=['Positive', 'Neutral', 'Negative'],
            linewidths=0.5)
plt.title('Confusion Matrix — Naive Bayes', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('Actual Label', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion Matrix saved as 'confusion_matrix.png'")

In [ ]:
# Test with a custom feedback message
def predict_sentiment(feedback_text):
    cleaned = clean_text(feedback_text)
    tokens = tokenize_and_lemmatize(cleaned)
    processed = ' '.join(tokens)
    vec = tfidf.transform([processed])
    prediction = nb_model.predict(vec)[0]
    return prediction

# Test examples
test_messages = [
    "The service was absolutely amazing and very helpful!",
    "Terrible experience. The team was slow and unresponsive.",
    "The service was okay, nothing special but got the job done."
]

print("=" * 55)
print("CUSTOM SENTIMENT PREDICTION TEST")
print("=" * 55)
for msg in test_messages:
    pred = predict_sentiment(msg)
    print(f"\nFeedback : {msg}")
    print(f"Predicted Sentiment: ➡️  {pred}")

In [ ]:
# Final Summary
print("=" * 55)
print("       TASK 7 — NLP PIPELINE SUMMARY")
print("=" * 55)
print(f"  Dataset Rows          : {len(df)}")
print(f"  Dataset Columns       : {len(df.columns)}")
print(f"  Positive Feedbacks    : {(df['Sentiment']=='Positive').sum()}")
print(f"  Neutral  Feedbacks    : {(df['Sentiment']=='Neutral').sum()}")
print(f"  Negative Feedbacks    : {(df['Sentiment']=='Negative').sum()}")
print(f"  TF-IDF Features       : {X.shape[1]}")
print(f"  Naive Bayes Accuracy  : {accuracy * 100:.2f}%")
print("=" * 55)
print("  ✅ All NLP Tasks Completed Successfully!")
print("=" * 55)